<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase27_HASH_FIXED_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 27 (Pinned-source hash correction)
## Same ImageWG revision · Dual source-hash verification · Synthetic Arm A staging

**Google Drive → Google Drive.** This notebook corrects the hash comparison that stopped the earlier schema repair. The Phase 27 inventory generated its SHA-256 from `Path.read_text()` (which normalizes CRLF line endings); the previous repair mistakenly compared that text hash with the SHA-256 of the **raw file bytes**. Those are different representations.

This correction checks **both** the expected raw-byte hash and the original Phase 27 text-inventory hash against the **same pinned ImageWG commit**. A source with different content or a changed revision fails the gate. It does not relax the mapping, terminology, patient-privacy or MI-CDM validation requirements.

The notebook finds Phase 26 JSON and its fully synthetic FHIR bundle in My Drive and writes the repaired Phase 27 artifacts beside them. Outputs remain synthetic **staging only**; this does **not** establish a complete Arm A target mapping, an approved Athena mapping, an official MI-CDM load, or clinical validity.

In [1]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:

# Phase 26 input discovery: safe against stale/moved Google Drive entries.
# You can override either individual file or the folder if automatic search misses them.
from pathlib import Path
import os, json, hashlib, subprocess, re, shutil, csv
from datetime import datetime, timezone
import pandas as pd

MYDRIVE = Path('/content/drive/MyDrive')

# OPTIONAL: set these to the CURRENT paths in My Drive (otherwise leave blank).
PHASE26_FOLDER_HINT = '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs'
PHASE26_JSON_OVERRIDE = ''
PHASE26_BUNDLE_OVERRIDE = ''

def _digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def _scan(base):
    '''Scan filenames, then *open* each file rather than stat() stale entries.'''
    json_paths, bundle_paths = [], []
    if not base.is_dir():
        return json_paths, bundle_paths
    def onerror(err):
        print('Skipping inaccessible Drive directory:', err)
    for folder, _, names in os.walk(str(base), onerror=onerror):
        for name in names:
            path = Path(folder) / name
            if name.startswith('ICHI2027_Phase26_SHAREABLE') and name.endswith('.json'):
                json_paths.append(path)
            elif name.startswith('ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE') and name.endswith('.json'):
                bundle_paths.append(path)
    return json_paths, bundle_paths

def _read_valid_phase26(path):
    try:
        with open(path, encoding='utf-8') as f:
            obj = json.load(f)
        if obj.get('phase') != '26_synthetic_fhir_r4_evidence_contract_gate':
            return None
        expected = obj.get('fhir_representation', {}).get('synthetic_bundle_sha256')
        if not expected:
            print('Skipping Phase 26 JSON without bundle SHA:', path)
            return None
        return obj
    except (OSError, ValueError, TypeError) as exc:
        print('Skipping missing, stale, or unreadable Drive entry:', path, exc)
        return None

def _choose_pair(json_paths, bundle_paths):
    verified_jsons = [(p, _read_valid_phase26(p)) for p in json_paths]
    verified_jsons = [(p, d) for p, d in verified_jsons if d is not None]
    valid_bundles = []
    for p in bundle_paths:
        try:
            digest = _digest(p)
            with open(p, encoding='utf-8') as f:
                bundle = json.load(f)
            if bundle.get('resourceType') == 'Bundle' and bundle.get('type') == 'collection':
                valid_bundles.append((p, digest))
        except (OSError, ValueError, TypeError) as exc:
            print('Skipping missing or unreadable FHIR bundle:', p, exc)

    # Prefer a matching file in the same folder; never pair unrelated runs.
    pairs = [
        (jp, bp)
        for jp, obj in verified_jsons
        for bp, digest in valid_bundles
        if digest == obj['fhir_representation']['synthetic_bundle_sha256']
    ]
    pairs.sort(key=lambda pair: (pair[0].parent == pair[1].parent, str(pair[0])), reverse=True)
    return pairs[0] if pairs else None, len(verified_jsons), len(valid_bundles)

# An explicit file path is respected. Otherwise, look in the folder first,
# then retry across My Drive if nothing usable is found.
jsons, bundles = [], []
if PHASE26_JSON_OVERRIDE.strip():
    jsons = [Path(PHASE26_JSON_OVERRIDE.strip())]
if PHASE26_BUNDLE_OVERRIDE.strip():
    bundles = [Path(PHASE26_BUNDLE_OVERRIDE.strip())]

if not (jsons and bundles):
    hinted_jsons, hinted_bundles = _scan(Path(PHASE26_FOLDER_HINT)) if PHASE26_FOLDER_HINT.strip() else ([], [])
    if not jsons: jsons = hinted_jsons
    if not bundles: bundles = hinted_bundles

selection, jcount, bcount = _choose_pair(jsons, bundles)
if selection is None:
    # Second pass: search the whole My Drive; do not call stat() on search hits.
    all_jsons, all_bundles = _scan(MYDRIVE)
    selection, jcount, bcount = _choose_pair(
        jsons if PHASE26_JSON_OVERRIDE.strip() else all_jsons,
        bundles if PHASE26_BUNDLE_OVERRIDE.strip() else all_bundles
    )

if selection is None:
    print(f'Valid Phase 26 shareable JSON files found: {jcount}')
    print(f'Readable Phase 26 synthetic FHIR bundles found: {bcount}')
    print('Drive root:', MYDRIVE)
    print('Folder hint:', PHASE26_FOLDER_HINT)
    raise FileNotFoundError(
        'Phase 27 needs BOTH ICHI2027_Phase26_SHAREABLE.json AND '
        'ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json from the SAME Phase 26 run. '
        'They must have a matching bundle SHA-256. Phase 24 JSON is not sufficient. '
        'Check the folder in Google Drive; if the bundle is missing, rerun Phase 26 '
        'with Runtime > Run all to save all three Phase 26 outputs, then rerun Phase 27.'
    )

PHASE26_JSON, PHASE26_BUNDLE = selection
WORKDIR = PHASE26_JSON.parent
print('Verified Phase 26 shareable:', PHASE26_JSON)
print('Verified matching FHIR bundle:', PHASE26_BUNDLE)
print('Phase 27 Drive output folder:', WORKDIR)


Verified Phase 26 shareable: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SHAREABLE.json
Verified matching FHIR bundle: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json
Phase 27 Drive output folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


In [3]:

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with open(PHASE26_JSON, "r", encoding="utf-8") as f:
    p26 = json.load(f)

assert p26["phase"] == "26_synthetic_fhir_r4_evidence_contract_gate"
assert p26["gates"]["phase25_verified"] is True
assert p26["gates"]["synthetic_FHIR_R4_representation_created"] is True
assert p26["gates"]["synthetic_required_evidence_reconstructable"] is True
assert p26["gates"]["synthetic_FHIR_contract_cases_passed"] is True
assert p26["gates"]["pyOMOP_ETL_executed"] is False
assert p26["gates"]["official_MICDM_load_completed"] is False
assert p26["gates"]["real_MRI_rows_processed"] == 0

assert sha256_file(PHASE26_BUNDLE) == p26['fhir_representation']['synthetic_bundle_sha256'], (
    'Phase 26 synthetic bundle SHA-256 mismatch: wrong or modified bundle.'
)
print("Phase 26 gate AND bundle hash verified.")
print("Phase 26 SHA256:", sha256_file(PHASE26_JSON))


Phase 26 gate AND bundle hash verified.
Phase 26 SHA256: 6cb3ef72e4d622d2a7330c7983c228f40a8e7aafecb5448395aace630f82bca4



## 1. Exact source pin and CSV parser self-test

Only the field-level CSV stored at the **exact previously recorded commit** can establish these MI-CDM column definitions. This revision does not reinterpret README paragraphs as field definitions.

Source: [ImageWG pinned field-level CSV](https://github.com/OHDSI/ImageWG/blob/8f0a19bb94e941bb335a097bc9851fab672621c7/inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv). If the pinned source is unavailable, it stops with an actionable error instead of inventing column names.


In [4]:

import io, unicodedata, urllib.request, time
PINNED_COMMIT = '8f0a19bb94e941bb335a097bc9851fab672621c7'
PINNED_RELATIVE_PATH = 'inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
EXPECTED_PINNED_INVENTORY_TEXT_SHA256 = '75de1b0ccab328742110157d464070a3a2c8cb37342c25f236ddfbeacb30c99b'  # sha256(read_text().encode('utf-8')) from Phase 27 inventory
EXPECTED_PINNED_RAW_SHA256 = 'af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4'  # exact pinned raw file bytes reported by Colab
PINNED_RAW_URL = f'https://raw.githubusercontent.com/OHDSI/ImageWG/{PINNED_COMMIT}/{PINNED_RELATIVE_PATH}'

def clean_str(value):
    return unicodedata.normalize('NFKC',str(value)).replace('\ufeff','').replace('\xa0',' ').strip()

def canonical_header(value):
    return re.sub(r'[^a-z0-9]','',clean_str(value).lower())

def parse_imagewg_fields(file_bytes):
    src = pd.read_csv(io.BytesIO(file_bytes),encoding='utf-8-sig',dtype=str,keep_default_na=False)
    headers = {canonical_header(c):c for c in src.columns}
    needed = {'cdmtablename','cdmfieldname','isrequired','cdmdatatype'}
    if not needed.issubset(headers):
        raise ValueError(f'Wrong source format. Found: {list(headers)}; need: {sorted(needed)}')
    rows=[]
    for _,r in src.iterrows():
        table=clean_str(r[headers['cdmtablename']]).lower()
        field=clean_str(r[headers['cdmfieldname']]).lower()
        if table not in ('image_occurrence','image_feature') or not field:
            continue
        rows.append({
            'table':table, 'field':field,
            'required':clean_str(r[headers['isrequired']]).lower() in ('yes','y','true'),
            'data_type':clean_str(r[headers['cdmdatatype']]),
            'source_relpath':PINNED_RELATIVE_PATH
        })
    fields=pd.DataFrame(rows)
    if fields.empty or fields.duplicated(['table','field']).any():
        raise ValueError('Empty or duplicate ImageWG field definitions; cannot freeze schema')
    defs={t: fields.loc[fields.table.eq(t),'field'].tolist() for t in ('image_occurrence','image_feature')}
    required={t:fields.loc[fields.table.eq(t) & fields.required,'field'].tolist() for t in defs}
    anchors={
      'image_occurrence': {'image_occurrence_id','person_id','procedure_occurrence_id','image_occurrence_date','image_study_uid','image_series_uid','modality_concept_id'},
      'image_feature': {'image_feature_id','person_id','image_occurrence_id','image_feature_concept_id','image_feature_type_concept_id','image_feature_event_id'}
    }
    for table,expected in anchors.items():
        if not expected.issubset(set(defs[table])):
            raise ValueError(f'Pinned {table} missing anchor fields: {sorted(expected-set(defs[table]))}')
    return fields,defs,required

# A deliberate regression fixture: BOM + NBSP + actual official CSV header names.
fixture=(
    '\ufeffcdmTableName,cdmFieldName,isRequired\xa0,cdmDatatype\xa0\n'
    + '\n'.join('IMAGE_OCCURRENCE,'+s+',Yes,integer' for s in [
         'image_occurrence_id','person_id','procedure_occurrence_id','image_occurrence_date',
         'image_study_UID','image_series_UID','modality_concept_id'])
    + '\n'+'\n'.join('IMAGE_FEATURE,'+s+',Yes,integer' for s in [
         'image_feature_id','person_id','image_occurrence_id',
         'image_feature_concept_id','image_feature_type_concept_id','image_feature_event_id'])
).encode('utf-8')
fixture_fields, fixture_defs, _ = parse_imagewg_fields(fixture)
assert len(fixture_fields) == 13
print('PASS: pinned CSV column-heading regression test (BOM + NBSP + cdmTableName/cdmFieldName)')


PASS: pinned CSV column-heading regression test (BOM + NBSP + cdmTableName/cdmFieldName)


In [5]:
# Optional local copy of the EXACT pinned CSV. Leave blank for automatic loading.
# Source must match BOTH the exact raw-blob hash and the original Phase 27
# text-inventory hash, otherwise the notebook STOPS without making output claims.
EXACT_PINNED_CSV_OVERRIDE = ''

source_bytes = None
source_origin = None
if EXACT_PINNED_CSV_OVERRIDE.strip():
    pinned_file = Path(EXACT_PINNED_CSV_OVERRIDE.strip())
    source_bytes = pinned_file.read_bytes()
    source_origin = 'explicit_copy_of_pinned_source:' + pinned_file.name
else:
    previous_checkout = Path('/content/OHDSI_ImageWG_phase27')
    if (previous_checkout / '.git').exists():
        actual_commit = subprocess.run(
            ['git', '-C', str(previous_checkout), 'rev-parse', 'HEAD'],
            capture_output=True, text=True
        )
        if actual_commit.returncode == 0 and actual_commit.stdout.strip() == PINNED_COMMIT:
            # Pull the blob from the pinned commit, NOT the mutable working tree.
            git_blob = subprocess.run(
                ['git', '-C', str(previous_checkout), 'show',
                 f'{PINNED_COMMIT}:{PINNED_RELATIVE_PATH}'],
                capture_output=True
            )
            if git_blob.returncode == 0:
                source_bytes = git_blob.stdout
                source_origin = 'git_blob_at_exact_pinned_commit'
    if source_bytes is None:
        try:
            req = urllib.request.Request(
                PINNED_RAW_URL, headers={'User-Agent': 'ICHI2027-Phase27-pinned-source'}
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                source_bytes = resp.read()
            source_origin = 'raw_github_url_at_exact_commit'
        except Exception as exc:
            raise RuntimeError(
                'Pinned ImageWG CSV unavailable. Set EXACT_PINNED_CSV_OVERRIDE '
                'to a byte-for-byte copy from the pinned commit.'
            ) from exc

# The previous inventory read text using pathlib.Path.read_text(); TextIOWrapper
# newline=None reproduces its CRLF/CR -> LF translation exactly.
def text_inventory_digest(blob):
    with io.TextIOWrapper(
        io.BytesIO(blob), encoding='utf-8', errors='ignore', newline=None
    ) as stream:
        text = stream.read()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

source_raw_sha256 = hashlib.sha256(source_bytes).hexdigest()
source_inventory_text_sha256 = text_inventory_digest(source_bytes)
print('Pinned commit             :', PINNED_COMMIT)
print('Source origin             :', source_origin)
print('Raw-byte SHA-256          :', source_raw_sha256)
print('Text-inventory SHA-256    :', source_inventory_text_sha256)

if source_raw_sha256 != EXPECTED_PINNED_RAW_SHA256 or source_inventory_text_sha256 != EXPECTED_PINNED_INVENTORY_TEXT_SHA256:
    raise RuntimeError(
        'Pinned ImageWG source verification failed. The two independently recorded '
        'expected digests must BOTH match the exact pinned CSV. '
        f'Raw expected={EXPECTED_PINNED_RAW_SHA256}; actual={source_raw_sha256}. '
        f'Text expected={EXPECTED_PINNED_INVENTORY_TEXT_SHA256}; actual={source_inventory_text_sha256}. '
        'Confirm the file came from the exact pinned ImageWG commit.'
    )

# Preserve original output key as the actual RAW source-file SHA-256.
source_sha256 = source_raw_sha256
field_rows, table_defs, required_fields = parse_imagewg_fields(source_bytes)
print('PASS: pinned ImageWG raw file AND Phase 27 text-inventory digests verified')
for table in table_defs:
    print(table.upper(), len(table_defs[table]),
          'fields; required:', required_fields[table])
assert all(len(table_defs[t]) >= 10 for t in table_defs)

Pinned commit             : 8f0a19bb94e941bb335a097bc9851fab672621c7
Source origin             : raw_github_url_at_exact_commit
Raw-byte SHA-256          : af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4
Text-inventory SHA-256    : 75de1b0ccab328742110157d464070a3a2c8cb37342c25f236ddfbeacb30c99b
PASS: pinned ImageWG raw file AND Phase 27 text-inventory digests verified
IMAGE_OCCURRENCE 11 fields; required: ['image_occurrence_id', 'person_id', 'procedure_occurrence_id', 'image_occurrence_date', 'image_study_uid', 'image_series_uid', 'modality_concept_id']
IMAGE_FEATURE 12 fields; required: ['image_feature_id', 'person_id', 'image_occurrence_id', 'image_feature_concept_id', 'image_feature_type_concept_id']



## 2. Faithful mapping requirements and synthetic staging

The pinned ImageWG `IMAGE_FEATURE` table links an image to a clinical-domain record. Quantitative volume values are therefore **not invented as columns in `IMAGE_FEATURE`**; this notebook creates a separate, explicitly provisional `MEASUREMENT` staging table and a documented, **not-yet-standardized** event link.

Missing required concepts (Athena), procedure linkage and synthetic DICOM UIDs are listed as explicit gaps. The original synthetic Phase 26 UID strings contain letters and are not attested DICOM UIDs; they will not be silently loaded into required DICOM UID fields.


In [6]:

with open(PHASE26_BUNDLE,encoding='utf-8') as f:
    fhir=json.load(f)
assert fhir.get('resourceType')=='Bundle' and fhir.get('type')=='collection'
idx={}
for entry in fhir['entry']:
    r=entry['resource']; key=f"{r.get('resourceType')}/{r.get('id')}"
    assert key not in idx, f'Duplicate FHIR resource: {key}'
    idx[key]=r
persons=sorted((r for r in idx.values() if r['resourceType']=='Patient'),key=lambda x:x['id'])
studies=sorted((r for r in idx.values() if r['resourceType']=='ImagingStudy'),key=lambda x:x['id'])
observations=sorted((r for r in idx.values() if r['resourceType']=='Observation'),key=lambda x:x['id'])
assert (len(persons),len(studies),len(observations))==(2,4,16)
assert all(f"{p['resourceType']}/{p['id']}" in idx for p in persons)
person_ids={p['id']:i for i,p in enumerate(persons,1)}
study_ids={s['id']:i for i,s in enumerate(studies,1)}
assert all(s.get('subject',{}).get('reference') in idx for s in studies)
print('Verified Phase26: 2 synthetic persons, 4 studies, 16 observations')

def ext_value(r,suffix,field):
    return next((e.get(field) for e in r.get('extension',[]) if e.get('url','').endswith(suffix)),None)

def valid_uid(s):
    return isinstance(s,str) and len(s)<=64 and bool(re.fullmatch(r'[0-9]+(?:\.[0-9]+)+',s))


Verified Phase26: 2 synthetic persons, 4 studies, 16 observations


In [7]:

# Official column sets always drive staging; never silently create imagined MI-CDM columns.
occ_rows=[]
feat_rows=[]
meas_rows=[]
provenance=[]

for s in studies:
    id_=s['id']; subj=s['subject']['reference'].split('/')[-1]
    rows=s.get('series',[])
    series_uid=rows[0].get('uid') if len(rows)==1 else None
    study_uid=next((v.get('value') for v in s.get('identifier',[]) if 'dicom' in v.get('system','').lower()),None)
    row={field:None for field in table_defs['image_occurrence']}
    row.update({
       'image_occurrence_id':study_ids[id_],
       'person_id':person_ids[subj],
       'image_occurrence_date':s.get('started','')[:10],
       'image_study_uid':study_uid if valid_uid(study_uid) else None,
       'image_series_uid':series_uid if valid_uid(series_uid) else None
    })
    # This synthetic FHIR source does not have a linked Procedure resource or Athena-approved modality concept.
    # A real/authorized source must support the missing fields before true Arm A execution.
    occ_rows.append(row)

for i,o in enumerate(observations,1):
    pid=o.get('subject',{}).get('reference','').split('/')[-1]
    focus=[x.get('reference','') for x in o.get('focus',[])]
    assert len(focus)==1 and focus[0] in idx, ('Unresolved source study',o['id'])
    sid=focus[0].split('/')[-1]
    s=idx[focus[0]]
    assert s['subject']['reference']==o['subject']['reference'],('Cross-patient imaging source',o['id'])
    q=o.get('valueQuantity',{})
    assert isinstance(q.get('value'),(int,float)) and q.get('system')=='http://unitsofmeasure.org'
    row={field:None for field in table_defs['image_feature']}
    row.update({
       'image_feature_id':i,
       'person_id':person_ids[pid],
       'image_occurrence_id':study_ids[sid],
       'image_feature_event_id':i  # provisional MEASUREMENT staging ID; cannot resolve field concept yet
    })
    feat_rows.append(row)
    meas_rows.append({
       'measurement_id':i,
       'person_id':person_ids[pid],
       'measurement_concept_id':None,
       'measurement_date':o.get('effectiveDateTime','')[:10],
       'measurement_type_concept_id':None,
       'value_as_number':q['value'],
       'unit_concept_id':None,
       'unit_source_value':q['unit'],
       'measurement_source_value':o.get('code',{}).get('coding',[{}])[0].get('code'),
       'fhir_observation_id':o['id']  # non-OMOP staging column; removed at actual load
    })
    devref=o.get('device',{}).get('reference')
    dev=idx.get(devref,{})
    prov=[r for r in idx.values() if r['resourceType']=='Provenance' and any(t.get('reference')==f"Observation/{o['id']}" for t in r.get('target',[]))]
    provenance.append({
      'synthetic_observation_id':o['id'],
      'staging_measurement_id':i,
      'staging_image_feature_id':i,
      'source_study_id':sid,
      'source_series_uid_original':s.get('series',[{}])[0].get('uid'),
      'synthetic_anatomy_laterality':o.get('bodySite',{}).get('coding',[{}])[0].get('code'),
      'synthetic_trace_state':ext_value(o,'trace-state','valueCode'),
      'synthetic_trace_rule_version':ext_value(o,'trace-rule-version','valueString'),
      'source_device_ref':devref,
      'source_device_name':dev.get('deviceName',[{}])[0].get('name'),
      'source_model_version':dev.get('version',[{}])[0].get('value'),
      'provenance_count':len(prov),
      'original_unit':q['unit'],
    })

occ=pd.DataFrame(occ_rows,columns=table_defs['image_occurrence'])
feat=pd.DataFrame(feat_rows,columns=table_defs['image_feature'])
meas=pd.DataFrame(meas_rows)
sidecar=pd.DataFrame(provenance)
assert (len(occ),len(feat),len(meas),len(sidecar))==(4,16,16,16)
assert set(feat.image_occurrence_id).issubset(set(occ.image_occurrence_id))
assert set(feat.image_feature_event_id)==set(meas.measurement_id)
assert all(len(o.get('focus',[]))==1 for o in observations)
print('PASS: Synthetic staging has 4 IMAGE_OCCURRENCE, 16 IMAGE_FEATURE, 16 MEASUREMENT rows and complete synthetic source joins.')


PASS: Synthetic staging has 4 IMAGE_OCCURRENCE, 16 IMAGE_FEATURE, 16 MEASUREMENT rows and complete synthetic source joins.


In [8]:

# Make missing required fields visible rather than presenting provisional rows as a faithful target load.
gaps=[]
for table,df in [('image_occurrence',occ),('image_feature',feat)]:
    id_col='image_occurrence_id' if table=='image_occurrence' else 'image_feature_id'
    for _,r in df.iterrows():
        for f in required_fields[table]:
            value=r.get(f)
            if value is None or (isinstance(value,float) and pd.isna(value)) or (isinstance(value,str) and not value.strip()):
                if f in ('image_study_uid','image_series_uid'):
                    reason='SYNTHETIC_SOURCE_UID_MISSING_OR_NOT_DICOM_UID'
                elif f.endswith('_concept_id'):
                    reason='STANDARD_CONCEPT_NOT_VALIDATED_IN_ATHENA'
                elif f=='procedure_occurrence_id':
                    reason='SOURCE_HAS_NO_LINKED_PROCEDURE_RECORD'
                else:
                    reason='REQUIRED_TARGET_FIELD_NOT_RESOLVED'
                gaps.append({'table':table,'staging_row_id':int(r[id_col]),'required_field':f,'reason':reason})

gapdf=pd.DataFrame(gaps,columns=['table','staging_row_id','required_field','reason'])
print('Pinned table-level schema gate: PASS')
print('Unresolved required-field instances:',len(gapdf))
print(gapdf.groupby(['table','required_field','reason'],dropna=False).size().to_string())
# Structural schema can pass while a faithful load remains blocked.
schema_verified=True
staging_created=True
faithful_armA_completed=bool(gapdf.empty) and False # no approved MEASUREMENT/procedure concepts or official load here
assert not faithful_armA_completed, 'This repair must never claim a completed Arm A database load.'


Pinned table-level schema gate: PASS
Unresolved required-field instances: 48
table             required_field                 reason                                       
image_feature     image_feature_concept_id       STANDARD_CONCEPT_NOT_VALIDATED_IN_ATHENA         16
                  image_feature_type_concept_id  STANDARD_CONCEPT_NOT_VALIDATED_IN_ATHENA         16
image_occurrence  image_series_uid               SYNTHETIC_SOURCE_UID_MISSING_OR_NOT_DICOM_UID     4
                  image_study_uid                SYNTHETIC_SOURCE_UID_MISSING_OR_NOT_DICOM_UID     4
                  modality_concept_id            STANDARD_CONCEPT_NOT_VALIDATED_IN_ATHENA          4
                  procedure_occurrence_id        SOURCE_HAS_NO_LINKED_PROCEDURE_RECORD             4



## 3. Save and verify Drive outputs

Files are labeled **REPAIRED** so the previous blocked Phase 27 outputs are not mistaken for these results. All patient identifiers in this notebook are synthetic. The field-level source CSV is public. The outputs do **not** include real ADNI records or real DICOM objects.


In [9]:

outputs={
 'shareable':WORKDIR/'ICHI2027_Phase27_REPAIRED_SHAREABLE.json',
 'field_audit':WORKDIR/'ICHI2027_Phase27_REPAIRED_PINNED_TARGET_FIELDS.csv',
 'mapping_contract':WORKDIR/'ICHI2027_Phase27_REPAIRED_MAPPING_CONTRACT.json',
 'required_gaps':WORKDIR/'ICHI2027_Phase27_REPAIRED_REQUIRED_FIELD_GAPS.csv',
 'occ_staging':WORKDIR/'ICHI2027_Phase27_ARM_A_IMAGE_OCCURRENCE_STAGING.csv',
 'feat_staging':WORKDIR/'ICHI2027_Phase27_ARM_A_IMAGE_FEATURE_STAGING.csv',
 'measurement_staging':WORKDIR/'ICHI2027_Phase27_ARM_A_MEASUREMENT_STAGING.csv',
 'synthetic_sidecar':WORKDIR/'ICHI2027_Phase27_ARM_A_SYNTHETIC_PROVENANCE_SIDECAR.csv',
}
field_rows.to_csv(outputs['field_audit'],index=False)
gapdf.to_csv(outputs['required_gaps'],index=False)
occ.to_csv(outputs['occ_staging'],index=False)
feat.to_csv(outputs['feat_staging'],index=False)
meas.to_csv(outputs['measurement_staging'],index=False)
sidecar.to_csv(outputs['synthetic_sidecar'],index=False)
contract={
  'omop_target':'CDM v5.5 (carried forward from Phase 27; individual core table schema not revalidated here)',
  'imagewg_pinned_commit':PINNED_COMMIT,
  'imagewg_source_relative_path':PINNED_RELATIVE_PATH,
  'imagewg_source_sha256':source_sha256,
  'imagewg_inventory_text_sha256':source_inventory_text_sha256,
  'imagewg_hash_verification':{'raw_blob_matches_expected':True,'text_inventory_matches_expected':True},
  'field_definitions':field_rows.to_dict(orient='records'),
  'required_fields':required_fields,
  'design_decisions':{
    'image_occurrence_granularity':'one synthetic series per ImagingStudy in this fixture; do not assume all real studies are single-series',
    'image_feature_links_to':'provisional MEASUREMENT staging ID, event-field concept remains unresolved',
    'numeric_volumes':'MEASUREMENT staging only, not invented IMAGE_FEATURE value columns',
    'source_reliability_and_laterality':'synthetic provenance sidecar until standards-supported destination confirmed',
    'athena_approved_concepts':False,
    'valid_real_dicoms_or_adni_uids':False,
    'actual_database_load':False
  }
}
with open(outputs['mapping_contract'],'w',encoding='utf-8') as f:json.dump(contract,f,indent=2)

shareable={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'27_repaired_pinned_schema_and_synthetic_armA_staging',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public synthetic-only staging; no ADNI records, RIDs, real series identifiers, or private Drive paths exported.',
 'phase26_sha256':sha256_file(PHASE26_JSON),
 'phase26_bundle_sha256':sha256_file(PHASE26_BUNDLE),
 'repaired_from_phase27_imagewg_commit':PINNED_COMMIT,
 'pinned_field_level_source_sha256':source_sha256,
 'pinned_field_level_inventory_text_sha256':source_inventory_text_sha256,
 'pinned_source_dual_hash_verified':True,
 'schema':{'image_occurrence_fields':len(table_defs['image_occurrence']),'image_feature_fields':len(table_defs['image_feature']), 'verified':True},
 'arm_registry':{
   'A':{'status':'SYNTHETIC_STAGING_CREATED_REQUIRED_FIELDS_UNRESOLVED','staging_rows':{'image_occurrence':len(occ),'image_feature':len(feat),'measurement':len(meas)},'faithful_target_mapping_completed':False},
   'B':{'status':'NOT_EXECUTED'},
   'C':{'status':'NOT_EXECUTABLE_ON_MRI_AS_IS','executed':False},
   'D':{'status':'SYNTHETIC_CONTRACT_PROTOTYPE_ONLY'}
 },
 'required_field_gap_count':len(gapdf),
 'required_field_gap_summary':[{'table':str(t),'field':str(f),'count':int(n)} for (t,f),n in gapdf.groupby(['table','required_field']).size().items()],
 'gates':{
  'phase26_verified':True,
  'same_phase27_imagewg_commit_verified':True,
  'exact_MICDM_field_level_schema_verified':True,
  'synthetic_armA_staging_created':True,
  'synthetic_armA_faithful_target_mapping_completed':False,
  'synthetic_armA_database_load_completed':False,
  'Athena_mappings_approved':False,
  'DICOM_UID_semantics_attested':False,
  'pyOMOP_ETL_executed':False,
  'official_MICDM_load_completed':False,
  'real_MRI_rows_processed':0,
  'true_four_arm_comparison_completed':False,
  'OMOP_target_cohort_equivalence_tested':False
 },
 'next_stage_gate':'Resolve procedure and standard concept bindings, syntactically valid but synthetic DICOM study/series UIDs, and standards-supported event field concept; independently validate OMOP 5.5 MEASUREMENT/PROCEDURE fields before a faithful executable synthetic Arm A and any Phase 28 cohort-equivalence claim.'
}
with open(outputs['shareable'],'w',encoding='utf-8') as f:json.dump(shareable,f,indent=2)

for name,path in outputs.items():
    assert path.exists() and path.stat().st_size>0,(name,path)
print('Saved and verified',len(outputs),'files in Google Drive:')
for path in outputs.values():print(' -',path.name)
print('SCHEMA REPAIRED: YES')
print('SYNTHETIC ARM A STAGING: YES')
print('FAITHFUL ARM A TARGET MAPPING: BLOCKED until required gaps resolved')


Saved and verified 8 files in Google Drive:
 - ICHI2027_Phase27_REPAIRED_SHAREABLE.json
 - ICHI2027_Phase27_REPAIRED_PINNED_TARGET_FIELDS.csv
 - ICHI2027_Phase27_REPAIRED_MAPPING_CONTRACT.json
 - ICHI2027_Phase27_REPAIRED_REQUIRED_FIELD_GAPS.csv
 - ICHI2027_Phase27_ARM_A_IMAGE_OCCURRENCE_STAGING.csv
 - ICHI2027_Phase27_ARM_A_IMAGE_FEATURE_STAGING.csv
 - ICHI2027_Phase27_ARM_A_MEASUREMENT_STAGING.csv
 - ICHI2027_Phase27_ARM_A_SYNTHETIC_PROVENANCE_SIDECAR.csv
SCHEMA REPAIRED: YES
SYNTHETIC ARM A STAGING: YES
FAITHFUL ARM A TARGET MAPPING: BLOCKED until required gaps resolved


In [10]:
# Final Drive write/read-back and source-integrity gate.
with open(outputs['shareable'], encoding='utf-8') as f:
    saved = json.load(f)
with open(outputs['mapping_contract'], encoding='utf-8') as f:
    saved_contract = json.load(f)
assert saved['pinned_source_dual_hash_verified'] is True
assert saved['pinned_field_level_source_sha256'] == EXPECTED_PINNED_RAW_SHA256
assert saved['pinned_field_level_inventory_text_sha256'] == EXPECTED_PINNED_INVENTORY_TEXT_SHA256
assert saved_contract['imagewg_source_sha256'] == EXPECTED_PINNED_RAW_SHA256
assert saved_contract['imagewg_inventory_text_sha256'] == EXPECTED_PINNED_INVENTORY_TEXT_SHA256
assert saved['gates']['exact_MICDM_field_level_schema_verified'] is True
assert saved['gates']['synthetic_armA_staging_created'] is True
assert saved['gates']['synthetic_armA_faithful_target_mapping_completed'] is False
assert saved['arm_registry']['C']['executed'] is False
print('PASS: repaired Phase 27 artifacts read back from Drive; dual hash gate verified.')
print('Send these two files to continue:')
print(' -', outputs['shareable'].name)
print(' -', outputs['required_gaps'].name)

PASS: repaired Phase 27 artifacts read back from Drive; dual hash gate verified.
Send these two files to continue:
 - ICHI2027_Phase27_REPAIRED_SHAREABLE.json
 - ICHI2027_Phase27_REPAIRED_REQUIRED_FIELD_GAPS.csv
